# Preprocessing of the Data. Download, import and get a first overview at the Data.

### Packages einladen

In [1]:
# import alle the needed packages
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import sklearn
import os
from dotenv import load_dotenv
from pathlib import Path


In [2]:
# Working Directory prüfen
cwd = Path.cwd()
print(f"Working Directory: {cwd}")

# Versuche .env im aktuellen oder übergeordneten Verzeichnis zu finden
env_path = None
for possible_path in [Path(".env"), Path("../.env"), cwd / ".env", cwd.parent / ".env"]:
    if possible_path.exists():
        env_path = possible_path
        print(f"✓ .env gefunden at: {env_path}")
        break

if env_path:
    load_dotenv(dotenv_path=env_path)
else:
    print("✗ Keine .env Datei gefunden!")

print("Data_path:", os.getenv("Data_path"))

data_base_path = os.getenv("Data_path")



Working Directory: c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\notebooks
✓ .env gefunden at: ..\.env
Data_path: C:\\Users\\User\\Documents\\Uni\\Master_Bauing\\WiSe25_26\\AI_in_Human_Water\\berlin-green-roofs\\data


In [3]:
# load shape file
shape_file_path = Path(data_base_path) / "green roofs 2020.shp"
green_roofs = gpd.read_file(shape_file_path)

print(green_roofs.columns)

Index(['gml_id', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gruen20_m2',
       'gint20_m2', 'gex20_m2', 'gruen20_p', 'gint20_p', 'gex20_p', 'geb_area',
       'nutz', 'ext', 'egeb_nutz', 'egruendach', 'eex_int', 'geometry'],
      dtype='str')


In [4]:
# lade die Daten über die Bezirke ein:
districts_shapes_file_path = Path(data_base_path) / "Bezirke.shp"
districts_data = gpd.read_file(districts_shapes_file_path)

# wandle die Daten über die Bezirke in einen pandas dataframe um eine attribute Tabelle zu bekommen
# districts_attribute_table = pd.DataFrame(districts_data.drop(columns="geometry"))
print(districts_data.columns)

# behalte nur die spalten "geometry" und "name" in den Bezirksdaten
districts_data = districts_data[["geometry","namgem"]]
print(districts_data.columns)

print(districts_data.head())

Index(['gml_id', 'name', 'gem', 'namgem', 'namlan', 'lan', 'geometry'], dtype='str')
Index(['geometry', 'namgem'], dtype='str')
                                            geometry  \
0  POLYGON ((390754.654 5825381.256, 390756.332 5...   
1  POLYGON ((396077.557 5817819.888, 396066.256 5...   
2  MULTIPOLYGON (((399003.49 5834202.526, 399004....   
3  POLYGON ((387115.47 5816898.439, 387112.266 58...   
4  POLYGON ((377248.57 5818041.669, 377119.245 58...   

                       namgem  
0                       Mitte  
1    Friedrichshain-Kreuzberg  
2                      Pankow  
3  Charlottenburg-Wilmersdorf  
4                     Spandau  


In [6]:
# lade die xml (.application datei ein)
xml_file_path = Path(data_base_path) / "Solarpotential.application"
with open(xml_file_path, "r") as file:
    xml_content = file.read()
print("XML Content:")
print(xml_content)

# lade die beiden layer in der application datei als shape file
solar_potential_layer_path = Path(data_base_path) / "Solarpotential.shp"
solar_potential_layer = gpd.read_file(solar_potential_layer_path)

print(solar_potential_layer.columns)

# lade die dachneigungs layer in der application datei als shape file
roof_slope_layer_path = Path(data_base_path) / "Dachneigung.shp"
roof_slope_layer = gpd.read_file(roof_slope_layer_path)
print(roof_slope_layer.columns)
print(green_roofs.columns)
# print anzahl spalten und zeilen der shape files
print(f"Green Roofs: {green_roofs.shape[0]} rows, {green_roofs.shape[1]} columns")
print(f"Districts: {districts_data.shape[0]} rows, {districts_data.shape[1]} columns")
print(f"Solar Potential: {solar_potential_layer.shape[0]} rows, {solar_potential_layer.shape[1]} columns")
print(f"Roof Slope: {roof_slope_layer.shape[0]} rows, {roof_slope_layer.shape[1]} columns")

XML Content:
<?xml version="1.0" encoding="UTF-8"?><wfs:WFS_Capabilities version="2.0.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.opengis.net/wfs/2.0" xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:fes="http://www.opengis.net/fes/2.0" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xs="http://www.w3.org/2001/XMLSchema" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd http://inspire.ec.europa.eu/schemas/inspire_dls/1.0 https://inspire.ec.europa.eu/schemas/inspire_dls/1.0/inspire_dls.xsd" xmlns:xml="http://www.w3.org/XML/1998/namespace" xmlns:inspire_dls="http://inspire.ec.europa.eu/schemas/inspire_dls/1.0" xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0" xmlns:ua_solarpotenzial_solarrechner="ua_solarpotenzial_solarrechner" updateSequence="512744"><ows:ServiceIdentification><ows:Title>Solarpotenzial - Sol

In [7]:
# behalte nur die wichtigsten spalten, um die shape files zu vereinfachen
green_roofs = green_roofs[["geometry", 'gruendach']]
print(green_roofs.columns)
solar_potential_layer = solar_potential_layer[["geometry", 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu', 'verschat_1', 'verschat_2', 'gebaeudefu',  'bauweise_s', 'id',]]
print(solar_potential_layer.columns)
dachneigung_layer = roof_slope_layer[["geometry", 'uid_gebaeu', 'ausrichtun', 'neigung', 'flaeche', 'id0',]]
print(dachneigung_layer.columns)

Index(['geometry', 'gruendach'], dtype='str')
Index(['geometry', 'anzahl_obe', 'ist_hochha', 'ist_denkma', 'verschattu',
       'verschat_1', 'verschat_2', 'gebaeudefu', 'bauweise_s', 'id'],
      dtype='str')
Index(['geometry', 'uid_gebaeu', 'ausrichtun', 'neigung', 'flaeche', 'id0'], dtype='str')


In [8]:
# Repariere ungültige Geometrien vor dem Clipping
print("Validiere Geometrien...")

green_roofs = green_roofs[green_roofs.geometry.is_valid]
dachneigung_layer = dachneigung_layer[dachneigung_layer.geometry.is_valid]
solar_potential_layer = solar_potential_layer[solar_potential_layer.geometry.is_valid]
#neukoelln = neukoelln[neukoelln.geometry.is_valid]

print(f"✓ Nach Validierung:")
print(f"  Green Roofs: {len(green_roofs)}")
print(f"  Dachneigung: {len(dachneigung_layer)}")
print(f"  Solar Potential: {len(solar_potential_layer)}")
#print(f"  Neukölln: {len(neukoelln)}")

# Falls noch Probleme: Fehlertolerant clippen
# green_roofs_neukoelln = gpd.clip(green_roofs, neukoelln, keep_geom_type=False)
# dachneigung_neukoelln = gpd.clip(dachneigung_layer, neukoelln, keep_geom_type=False)
# solar_potential_neukoelln = gpd.clip(solar_potential_layer, neukoelln, keep_geom_type=False)

Validiere Geometrien...
✓ Nach Validierung:
  Green Roofs: 629488
  Dachneigung: 1481759
  Solar Potential: 529946


Da die Dächer aufgeteilt sind in verschiedene Neigungen, müssen sie zunächst jedem Gebäude zugeordnet werden.
Im Anschluss wird entschieden, ob ein Gebäude einen Flachdachanteil hat, der Groß genug für ein gründach ist.

In [7]:
# ============================================================
# NEUER ANSATZ: Gruppiere nach uid_gebaeu, nutze vorhandene flaeche
# ============================================================

print("="*70)
print("Berechne Anteil geeigneter Fläche pro Gebäude (uid_gebaeu)...")
print("="*70)

SLOPE_THRESHOLD = 15.0

# Definiere "geeignet"
dachneigung_layer['is_suitable'] = dachneigung_layer['neigung'] <= SLOPE_THRESHOLD

# Gruppiere nach uid_gebaeu und aggregiere
building_results_v2 = dachneigung_layer.groupby('uid_gebaeu', as_index=False).agg({
    'flaeche': 'sum',  # Gesamtfläche des Gebäudes
    'neigung': 'mean',  # Durchschnittliche Neigung
    'ausrichtun': lambda x: x.mode()[0] if len(x.mode()) > 0 else None  # Häufigste Ausrichtung
}).rename(columns={
    'flaeche': 'total_roof_area',
    'neigung': 'avg_slope',
    'ausrichtun': 'dominant_orientation'
})

# Berechne suitable_area (nur Segmente mit neigung <= 15°)
suitable_area_per_building = dachneigung_layer[dachneigung_layer['is_suitable']].groupby('uid_gebaeu')['flaeche'].sum()
building_results_v2['suitable_roof_area'] = building_results_v2['uid_gebaeu'].map(suitable_area_per_building).fillna(0)

# Berechne Anteil
building_results_v2['suitable_share_pct'] = np.where(
    building_results_v2['total_roof_area'] > 0,
    (building_results_v2['suitable_roof_area'] / building_results_v2['total_roof_area']) * 100,
    0.0
)

print(f"✓ {len(building_results_v2)} Gebäude verarbeitet")
print(f"\nStatistik:")
print(f"  Durchschnittlicher Anteil: {building_results_v2['suitable_share_pct'].mean():.1f}%")
print(f"  Gebäude mit >75% geeigneter Fläche: {(building_results_v2['suitable_share_pct'] > 75).sum()}")
print(f"\nVorschau (Top 10):")
print(building_results_v2.nlargest(10, 'suitable_share_pct')[['uid_gebaeu', 'total_roof_area', 'suitable_roof_area', 'suitable_share_pct']].to_string(index=False))

Berechne Anteil geeigneter Fläche pro Gebäude (uid_gebaeu)...
✓ 529946 Gebäude verarbeitet

Statistik:
  Durchschnittlicher Anteil: 43.2%
  Gebäude mit >75% geeigneter Fläche: 205596

Vorschau (Top 10):
      uid_gebaeu  total_roof_area  suitable_roof_area  suitable_share_pct
DEBE00YY10z0001A           252.06              252.06               100.0
DEBE00YY115000B7           156.83              156.83               100.0
DEBE00YY1170005P          2226.53             2226.53               100.0
DEBE00YY1180000H           182.03              182.03               100.0
DEBE00YY11B0005L           184.34              184.34               100.0
DEBE00YY11B0005z           121.42              121.42               100.0
DEBE00YY11B0007U           170.23              170.23               100.0
DEBE00YY11C0004e           134.09              134.09               100.0
DEBE00YY11C0004f            59.15               59.15               100.0
DEBE00YY11I000Dv           104.62              104.62    

In [1]:
# ============================================================
# EXPORTIERE ALS GEODATAFRAME FÜR QGIS
# ============================================================

print("\n" + "="*70)
print("Konvertiere zu GeoDataFrame und exportiere für QGIS...")
print("="*70)

# Merge mit Geometrien aus dachneigung_layer
# Fasse ALLE Geometrien pro uid_gebaeu zusammen (mit Fehlerbehandlung)
print("Fasse Geometrien zusammen (unary_union)...")

def safe_union(geom_series):
    """Verbinde Geometrien, repariere ungültige vorher"""
    try:
        # Validiere Geometrien
        valid_geoms = geom_series[geom_series.is_valid]
        if len(valid_geoms) == 0:
            return None
        # Vereinige zu einem Polygon
        return valid_geoms.union_all()
    except Exception as e:
        print(f"  ⚠ Fehler bei union: {e}")
        return None

geometry_per_building = dachneigung_layer.groupby('uid_gebaeu').agg({
    'geometry': safe_union  # Alle Segmente zu einem Polygon vereinigen
}).reset_index()

print(f"✓ Geometrien zusammengefügt: {len(geometry_per_building)} Gebäude")

# Merge building_results_v2 mit Geometrien
building_results_v2_geo = building_results_v2.merge(
    geometry_per_building,
    left_on='uid_gebaeu',
    right_on='uid_gebaeu',
    how='left'
)

# Konvertiere zu GeoDataFrame
building_results_v2_geo = gpd.GeoDataFrame(
    building_results_v2_geo,
    geometry='geometry',
    crs=dachneigung_layer.crs
)

print(f"✓ {len(building_results_v2_geo)} Features als GeoDataFrame konvertiert")
print(f"  CRS: {building_results_v2_geo.crs}")
print(f"  Spalten: {list(building_results_v2_geo.columns)}")

# Speichern als Shapefile
output_shapefile = Path(data_base_path) / "gebaeude_gruendach_potential_v2.shp"
building_results_v2_geo.to_file(output_shapefile)
print(f"\n✓ Shapefile gespeichert: {output_shapefile.name}")

# Speichern als GeoPackage (bessere Alternative)
output_gpkg = Path(data_base_path) / "gebaeude_gruendach_potential_v2.gpkg"
building_results_v2_geo.to_file(output_gpkg, driver='GPKG')
print(f"✓ GeoPackage gespeichert: {output_gpkg.name}")

print(f"\n✓ Beide Dateien sind bereit für QGIS-Import!")


Konvertiere zu GeoDataFrame und exportiere für QGIS...
Fasse Geometrien zusammen (unary_union)...



Konvertiere zu GeoDataFrame und exportiere für QGIS...
Fasse Geometrien zusammen (unary_union)...



KeyboardInterrupt



In [9]:
# lade die dateien zum weiterarbeiten hier wieder rein
building_results_v2_geo_path = Path(data_base_path) / "gebaeude_gruendach_potential_v2.shp"
building_results_v2_geo = gpd.read_file(building_results_v2_geo_path)

In [ ]:
# ============================================================
# VERSCHNEIDE DIE DREI LAYER
# ============================================================

print("="*70)
print("Verschneide die drei Layer...")
print("="*70)

# # Wir erstellen Punkte aus den Dachsegmenten
green_roofs_points = green_roofs.copy()
green_roofs_points['geometry'] = green_roofs_points['geometry'].representative_point()

print(f"spalten in green_roofs_points: {green_roofs_points.columns}")
# SCHRITT 1: Spatial Join mit green_roofs (intersects)
print("\nSchritt 1: Verschneide building_results_v2_geo mit green_roofs...")
merged_layer = gpd.sjoin(
    building_results_v2_geo,
    green_roofs_points,
    how='left',  # Left join: behalte alle building_results, auch wenn kein match
    predicate='contains'
)
print(f"✓ {len(merged_layer)} Features nach green_roofs-Merge")
print(f"  Spalten: {list(merged_layer.columns)}")

# Entferne doppelte Spalten (index_right ist von sjoin)
if 'index_right' in merged_layer.columns:
    merged_layer = merged_layer.drop(columns=['index_right'])

# bereite die Punkte für den nächsten Join vor (falls nötig)
solar_potential_layer_points = solar_potential_layer.copy()
solar_potential_layer_points['geometry'] = solar_potential_layer_points['geometry'].representative_point()
print(solar_potential_layer_points.columns)
# SCHRITT 2: Spatial Join mit solar_potential_layer  
print("\nSchritt 2: Verschneide Ergebnis mit solar_potential_layer...")
merged_layer = gpd.sjoin(
    merged_layer,
    solar_potential_layer_points,
    how='left',  # Left join: behalte alle, auch wenn kein match
    predicate='contains'
)
print(f"✓ {len(merged_layer)} Features nach solar_potential-Merge")

# Entferne doppelte Spalten
if 'index_right' in merged_layer.columns:
    merged_layer = merged_layer.drop(columns=['index_right'])

# Bereinige Spalten (removed geometry columns falls mehrfach)
print(f"\nGesamt-Spalten: {len(merged_layer.columns)}")
print(f"Spalten: {list(merged_layer.columns)}")

# SCHRITT 3: Exportiere als Shapefile
print("\nSchritt 3: Exportiere als Shapefile...")
output_merged = Path(data_base_path) / "merged_gebaeude_gruendach_solar.shp"
merged_layer.to_file(output_merged)
print(f"✓ Shapefile gespeichert: {output_merged.name}")
print(f"  Anzahl Features: {len(merged_layer)}")
print(f"  Anzahl Spalten: {len(merged_layer.columns)}")

# Exportiere auch als GeoPackage
output_merged_gpkg = Path(data_base_path) / "merged_gebaeude_gruendach_solar.gpkg"
merged_layer.to_file(output_merged_gpkg, driver='GPKG')
print(f"✓ GeoPackage gespeichert: {output_merged_gpkg.name}")

print(f"\n✓ Verschneidung abgeschlossen! Bereit für QGIS-Kontrolle.")

Verschneide die drei Layer...
